In [ ]:
import argparse
import csv
import json
import os

import numpy as np

from motor_lib import (BATTERY_V, LINEAR_FIT_MIN_PWM, WHEELS, fit_all,
                       fit_wheel_model, read_raw_log, split_runs)

HERE = os.path.dirname(os.path.abspath('mtr_idf.txt'))


def load(raw, battery):
    runs = split_runs(read_raw_log(raw), battery)
    fits = fit_all(runs)
    models = {w: fit_wheel_model(fits[w]) for w in fits}
    return runs, fits, models


def write_csv(fits, path):
    cols = ["wheel", "pwm", "U", "n", "dt_ms", "moved", "w_ss", "w_ss_std",
            "T_m", "T_m_std", "rmse_deg", "w_tail", "note"]
    with open(path, "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(cols)
        for w in fits:
            for r in fits[w]:
                wr.writerow([getattr(r, c) if not isinstance(getattr(r, c), float)
                             else f"{getattr(r, c):.6g}" for c in cols])



def main():
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--raw", default=os.path.join(HERE, "mtr_idf.txt"))
    ap.add_argument("--out", default=os.path.join(HERE, "results"))
    ap.add_argument("--battery", type=float, default=None,
                    help=f"measured battery voltage, V (default: nominal {BATTERY_V})")
    args, unknown = ap.parse_known_args()
    os.makedirs(args.out, exist_ok=True)

    logged = args.battery is not None
    battery = args.battery if logged else BATTERY_V
    _runs, fits, models = load(args.raw, battery)
    write_csv(fits, os.path.join(args.out, "step_fits.csv"))

    for w, m in models.items():
        print(f"wheel {w} ({WHEELS[w]['name']}): k_e = {m.k_e:.3f} V s/rad "
              f"(R2 {m.k_e_r2:.3f}), median T_m = {m.T_m_median:.3f} s, "
              f"dead zone inside {m.dz_bound[0]:+.2f}..{m.dz_bound[1]:+.2f} V")
    print(f"results written to {os.path.relpath(args.out)}")


if __name__ == "__main__":
    main()

wheel 1 (left): k_e = 0.701 V s/rad (R2 0.974), median T_m = 0.204 s, dead zone inside -0.56..+0.56 V
wheel 2 (right): k_e = 0.678 V s/rad (R2 0.973), median T_m = 0.260 s, dead zone inside -0.56..+0.56 V
results written to results
